[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/ch4/lesson3.ipynb)

# La cobertura terrestre a lo largo del tiempo

¿Cómo han cambiado el uso del suelo y la cobertura terrestre con el paso del tiempo?

In [ ]:
import folium
import ee
import geemap
import geopandas as gpd

ee.Authenticate()

# Escribe aquí el ID de tu proyecto, entre comillas
ee.Initialize(project="emerge-lessons")

In [ ]:
def add_ee_layer(self, ee_image_object, vis_params, name):
    """
    Agrega a un mapa de Folium una capa de teselas
    procedente de una imagen de Earth Engine.
    """
    map_id_dict = ee.Image(
        ee_image_object
    ).getMapId(vis_params)

    folium.raster_layers.TileLayer(
        tiles=map_id_dict["tile_fetcher"].url_format,
        attr=(
            'Map Data &copy; '
            '<a href="https://earthengine.google.com/">'
            "Google Earth Engine</a>"
        ),
        name=name,
        overlay=True,
        control=True
    ).add_to(self)


folium.Map.add_ee_layer = add_ee_layer

In [ ]:
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="Cartodb dark_matter",
    zoom_start=7
)

Utilizaremos datos de cobertura terrestre y uso del suelo del conjunto [USFS Landscape Change Monitoring System v2024.10 (CONUS and OCONUS)](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10), disponible en Google Earth Engine.

In [ ]:
landcover_1985 = (
    ee.ImageCollection(
        "USFS/GTAC/LCMS/v2024-10"
    )
    .filterDate("1985", "1986")
    .filter('study_area == "CONUS"')
    .first()
)

palette = [
    "fbff97",
    "e6558b",
    "004e2b",
    "9dbac5",
    "a6976a",
    "1b1716"
]

visual = {
    "min": 1,
    "max": 6,
    "palette": palette
}

map.add_ee_layer(
    landcover_1985.select("Land_Use"),
    visual,
    "Uso del suelo en 1985"
)

In [ ]:
display(map)

In [ ]:
landcover_2024 = (
    ee.ImageCollection(
        "USFS/GTAC/LCMS/v2024-10"
    )
    .filterDate("2024", "2025")
    .filter('study_area == "CONUS"')
    .first()
)

map.add_ee_layer(
    landcover_2024.select("Land_Use"),
    visual,
    "Uso del suelo en 2024"
)

folium.LayerControl(
    collapsed=False
).add_to(map)

display(map)

En la esquina superior derecha, haz clic en la casilla junto a **Uso del suelo en 2024** para mostrar u ocultar esa capa y comparar los cambios ocurridos entre 1985 y 2024.

Significado de los colores, según el [catálogo de datos](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10#bands):

$$\color{#fbff97}Agricultura$$
$$\color{#e6558b}Zona\space desarrollada$$
$$\color{#004e2b}Bosque$$
$$\color{#9dbac5}Otra\space categoría$$
$$\color{#a6976a}Pastizal\space o\space zona\space de\space pastoreo$$

Ahora mostraremos la cobertura terrestre, en lugar del uso del suelo.

In [ ]:
# Crear un mapa nuevo
map = folium.Map(
    location=[28.263363, -83.497652],
    tiles="Cartodb dark_matter",
    zoom_start=7
)

palette = [
    "004e2b",
    "009344",
    "61bb46",
    "acbb67",
    "8b8560",
    "cafd4b",
    "f89a1c",
    "8fa55f",
    "bebb8e",
    "e5e98a",
    "ddb925",
    "893f54",
    "e4f5fd",
    "00b6f0",
    "1b1716"
]

visual = {
    "min": 1,
    "max": 15,
    "palette": palette
}

# Agregar la cobertura terrestre de 1985 y 2024
map.add_ee_layer(
    landcover_1985.select("Land_Cover"),
    visual,
    "Cobertura terrestre en 1985"
)

map.add_ee_layer(
    landcover_2024.select("Land_Cover"),
    visual,
    "Cobertura terrestre en 2024"
)

folium.LayerControl(
    collapsed=False
).add_to(map)

display(map)

$$\color{#004e2b}Árboles$$
$$\color{#61bb46}Mezcla\space de\space arbustos\space y\space árboles$$
$$\color{#acbb67}Mezcla\space de\space pasto,\space herbáceas\space y\space árboles$$
$$\color{#8b8560}Mezcla\space de\space terreno\space descubierto\space y\space árboles$$
$$\color{#f89a1c}Arbustos$$
$$\color{#8fa55f}Mezcla\space de\space pasto,\space herbáceas\space y\space arbustos$$
$$\color{#bebb8e}Mezcla\space de\space terreno\space descubierto\space y\space arbustos$$
$$\color{#e5e98a}Pasto,\space plantas\space herbáceas\space o\space hierbas$$
$$\color{#ddb925}Mezcla\space de\space terreno\space descubierto,\space pasto\space y\space herbáceas$$
$$\color{#893f54}Terreno\space descubierto\space o\space superficie\space impermeable$$
$$\color{#e4f5fd}Nieve\space o\space hielo$$

## Crear una animación temporal

In [ ]:
landcover = (
    ee.ImageCollection(
        "USFS/GTAC/LCMS/v2024-10"
    )
    .filter('study_area == "CONUS"')
    .select("Land_Use")
)

point = ee.Geometry.Point(
    -81.660044,
    28.473813
)

region = point.buffer(
    distance=100000
)

images = geemap.create_timeseries(
    landcover,
    "1985",
    "2024",
    region,
    frequency="year",
    reducer="first"
)

images

In [ ]:
Map = geemap.Map()

# Incluir una etiqueta para cada año, desde 1985 hasta 2024
labels = [
    str(year)
    for year in range(1985, 2025)
]

Map.addLayer(
    images,
    visual,
    "Uso del suelo"
)

Map.add_time_slider(
    images,
    visual,
    time_interval=2,
    labels=labels
)

Map.setCenter(
    -81.660044,
    28.473813,
    zoom=8
)

Map

En Google Colab, haz clic en la flecha `>` para ver una animación de los cambios en el uso del suelo a lo largo de los años. La animación puede tardar algunos segundos en cargarse.

In [ ]:
fl = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/raw/"
    "refs/heads/main/docs/data/florida_boundary.geojson"
)[["geometry"]]

region = geemap.geopandas_to_ee(fl)

timelapse = geemap.modis_ndvi_timelapse(
    region,
    out_gif="ndvi.gif",
    data="Terra",
    band="NDVI",
    start_date="2000-01-01",
    end_date="2024-12-31",
    frames_per_second=3,
    title="Animación temporal del NDVI de MODIS"
)

In [ ]:
geemap.show_image(timelapse)

![Animación temporal de la vegetación de Florida](https://github.com/geo-di-lab/emerge-lessons/blob/main/docs/ch4/ndvi.gif?raw=1)

## Referencias

- [Creación de animaciones temporales con geemap](https://book.geemap.org/chapters/09_timelapse.html)
- [Visualización de imágenes en Google Earth Engine](https://developers.google.com/earth-engine/guides/image_visualization#colab-python_2)